In [ ]:
# Tree Buffer Counts
# This notebook intersects the noise street network with the street-tree and park-tree geopackages using 10 m and 20 m buffers.
#
# The output is a separate CSV with `fid`, `TRAM`, separate counts for street trees and park trees, and total tree counts in both buffers.

In [ ]:
import geopandas as gpd
import pandas as pd
import os
import matplotlib.pyplot as plt

print("Loading layers...")
noise_streets = gpd.read_file("../../layers/BCN_noise_streets.gpkg")
street_trees = gpd.read_file("../../layers/BCN_STREET_TREES.gpkg")
park_trees = gpd.read_file("../../layers/BCN_PARK_TREES.gpkg")

noise_streets = noise_streets.reset_index(drop=True).copy()
noise_streets["fid"] = noise_streets.index.astype(int)
noise_streets["TRAM"] = noise_streets["TRAM"].astype(str)

if street_trees.crs != noise_streets.crs:
    street_trees = street_trees.to_crs(noise_streets.crs)
if park_trees.crs != noise_streets.crs:
    park_trees = park_trees.to_crs(noise_streets.crs)

print("Noise streets CRS:", noise_streets.crs)
print("Street trees CRS:", street_trees.crs)
print("Park trees CRS:", park_trees.crs)
print("Number of noise streets:", len(noise_streets))
print("Number of street trees:", len(street_trees))
print("Number of park trees:", len(park_trees))

Loading layers...
Noise streets CRS: EPSG:25831
Street trees CRS: EPSG:25831
Park trees CRS: EPSG:25831
Number of noise streets: 15115
Number of street trees: 145706
Number of park trees: 32776


,fid,TRAM,street_tree_count_10m,park_tree_count_10m,street_tree_count_20m,park_tree_count_20m
0,0,T04719W,12,0,16,11
1,1,T19941Z,11,0,18,0
2,2,T18111R,5,0,8,0
3,3,T03222Y,1,0,1,0
4,4,T17625I,24,0,28,0


Final dataset shape: (15115, 6)
Dataset successfully exported to: ../../data/processed\tree_buffer_counts.csv


In [ ]:
# Count tree points inside 10 m and 20 m buffers around each noise street.
def count_trees_in_buffer(streets_gdf, trees_gdf, buffer_m, count_column):
    buffered_streets = streets_gdf[["fid", "TRAM", "geometry"]].copy()
    buffered_streets["geometry"] = buffered_streets.geometry.buffer(buffer_m)

    joined = gpd.sjoin(buffered_streets, trees_gdf[["geometry"]].copy(), how="left", predicate="intersects")
    counts = (
        joined.groupby("fid")["index_right"]
        .count()
        .rename(count_column)
        .reset_index()
    )

    return counts

street_tree_counts_10m = count_trees_in_buffer(noise_streets, street_trees, 10, "street_tree_count_10m")
park_tree_counts_10m = count_trees_in_buffer(noise_streets, park_trees, 10, "park_tree_count_10m")
street_tree_counts_20m = count_trees_in_buffer(noise_streets, street_trees, 20, "street_tree_count_20m")
park_tree_counts_20m = count_trees_in_buffer(noise_streets, park_trees, 20, "park_tree_count_20m")

tree_buffer_counts = noise_streets[["fid", "TRAM"]].copy()
tree_buffer_counts = tree_buffer_counts.merge(street_tree_counts_10m, on="fid", how="left")
tree_buffer_counts = tree_buffer_counts.merge(park_tree_counts_10m, on="fid", how="left")
tree_buffer_counts = tree_buffer_counts.merge(street_tree_counts_20m, on="fid", how="left")
tree_buffer_counts = tree_buffer_counts.merge(park_tree_counts_20m, on="fid", how="left")

tree_buffer_counts["total_tree_count_10m"] = tree_buffer_counts["street_tree_count_10m"] + tree_buffer_counts["park_tree_count_10m"]
tree_buffer_counts["total_tree_count_20m"] = tree_buffer_counts["street_tree_count_20m"] + tree_buffer_counts["park_tree_count_20m"]

tree_buffer_counts = tree_buffer_counts.fillna(0)
count_columns = [
    "street_tree_count_10m",
    "park_tree_count_10m",
    "street_tree_count_20m",
    "park_tree_count_20m",
    "total_tree_count_10m",
    "total_tree_count_20m",
]
tree_buffer_counts[count_columns] = tree_buffer_counts[count_columns].astype(int)

display(tree_buffer_counts.head())
print(f"Final dataset shape: {tree_buffer_counts.shape}")

In [ ]:
# Visual 1: Barcelona street trees and park trees with two different colors.
fig, ax = plt.subplots(figsize=(12, 12))
noise_streets.plot(ax=ax, color="lightgrey", linewidth=0.6, alpha=0.55)
street_trees.plot(ax=ax, color="#2E8B57", markersize=4, alpha=0.7, label="Street trees")
park_trees.plot(ax=ax, color="#D97706", markersize=4, alpha=0.7, label="Park trees")
ax.set_title("Barcelona Street Trees and Park Trees")
ax.set_axis_off()
ax.legend(loc="upper right")
plt.show()

# Visual 2: Noise streets influenced by trees within the 20 m buffer.
tree_influence_20m = noise_streets[["fid", "TRAM", "geometry"]].merge(
    tree_buffer_counts[["fid", "total_tree_count_20m"]],
    on="fid",
    how="left",
)

fig, ax = plt.subplots(figsize=(12, 12))
tree_influence_20m.plot(
    ax=ax,
    column="total_tree_count_20m",
    cmap="YlGn",
    linewidth=2,
    legend=True,
    legend_kwds={"label": "Trees within 20 m buffer"},
)
ax.set_title("Noise Streets Influenced by Nearby Trees")
ax.set_axis_off()
plt.show()

In [ ]:
# Export the final counts table.
output_dir = "../../data/processed"
os.makedirs(output_dir, exist_ok=True)

final_path = os.path.join(output_dir, "tree_buffer_counts.csv")
tree_buffer_counts.to_csv(final_path, index=False)

print(f"Dataset successfully exported to: {final_path}")